In [0]:
%%capture
!pip install optuna

In [0]:
%%capture
!pip install catboost

In [0]:
import sys, os
import pytz
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpyme/'))

# Install dependecies
packages = ['xgboost',
            'lightgbm',
            'catboost',
            'openpyx1',
            'category_encoders',
            'fsspec']

for package in packages:
    os.system(f"pip install {package}--quiet")
from decorators import *
#from auxiliary_functions import *

from sklearn.model_selection import StratifiedKFold, KFold,StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from auxiliary_functions import spark_to_pandas

#from matplotlib_venn import venn2
from sklearn import preprocessing
import matplotlib.pyplot as plt
import pickle
import seaborn as sns from glob
import glob #from tqdm
import tqdm
import pandas as pd
import numpy as np

%matplotlib inline
import itertools
import warnings
import os
import gc
pd.options.display.max_columns=None
warnings.filterwarnings("ignore")

from pyspark.sql.functions import col, count, date_format
from pyspark.sql import functions as F
from pyspark.sql.functions import col, countDistinct
import sys, os
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpyme/')) 
#sys.path.append(os.path.abspath('/Workspace/Users/rodrigoaasencios@bcp.com.pe/202404_SEGMENTACION_ORIGINACION'))

from utils import *
from write_to storage import *
from write_to_storage import pd_read chunks ###
from pyspark.sql import SparkSession
from catboost import CatBoostClassifier, Pool

In [0]:

import sys, os
sys.path.append(os.path.abspath('/Workspace/Repos/rodrigoaasencios@bcp.com.pe/fabpy=e/'))
from utils import *
from write_to storage import *
from pyspark.sql import SparkSession


In [0]:
# Install dependecies
packages = ['xgboost',
            'lightgbm',
            "catboost',
            'openpyx1',
            'category_encoders',            
            "fsspec']
            
for package in packages:
    os.system(f"pip install {package}--quiet")
from decorators import *
#from auxiliary_functions import
from auxiliary_functions import spark_to_pandas

#CONSTRUCCION DE VARIABLES QUE SALEN DE TABLAS SCORE

##VARIABLES de la tabla "hm_scorepreaprobacionapppyme"

In [0]:
# =====================================================================================
# VARIABLES CON UN MES DE DESFASE: 
# Las siguientes variables ya fueron construidas con información de CODMES_DATA 202505 
# para ser utilizadas en la actualización de la tabla BHV correspondiente al 
# CODMES_DATA - 202506. Es decir, estas variables tienen un desfase de 1 mes 
# respecto al periodo de actualización.
# =====================================================================================

##RECABLEO DE VARIABLES INPUT

In [0]:
## ESTAS VARIABLES SE VAN A RECABLEAR 
# rev_APP SCORE APROB PYME = spark.table("catalog lhcl prod_bcp.bcp ddv_rbmbcapym_apppyme_vu. hm_scorepreaprobacionapppyme")\
    #.select(F.col("CODMES").alias("CODMES_0"), "codclaveunicocli", F.col("atrasomax_crnenr_12_rl").alias ("APP_SCORE_APROB_PYME atrasomax_crnenr 12 rl"), 
#    F.col("meses_pmsavmf_24_100_rl").alias 
# ("APP_SCORE_APROB_PYME_meses_pmsavmf 24 100 rl"), 
#    F.col("montoade_act_max6 s hip rl").alias ("APP_SCORE_APROB_PYME_montoade_act_max6_s_hip_rl"), 
#    F.col("sf_num_cal_cpp24_ag").alias("APP_SCORE_APROB_PYME_sf_num_cal_cpp24_ag"),
####
#   F.col("utl_3_rl").alias("APP_SCORE_APROB_PYME_ut1_3_rl"), ==== 
#   F.col("act_eco_fin").alias("APP_SCORE_APROB_PYME_act_eco_fin"), 
#   F.col("edad_fin").alias("APP_SCORE_APROB_PYME_edad_fin"), 
#   F.col("meses_activo_sf_bu_ma6_0_ag").alias ("APP_SCORE_APROB_PYME_meses_activo_sf_bu_ma6_0_ag"),
#   )/
#  .filter(F.col("codmes")==202505)\ 
#  .distinct()
 
# rev_APP_SCORE_APROB_PYME = rev_APP_SCORE_APROB_PYME.withColumn('codmes', add_codmes_spark ('CODMES_0', +1)) 
# rev_APP_SCORE_APROB_PYME = rev_APP_SCORE_APROB_PYME.drop("CODMES_0")

In [0]:
sp_clientes0 = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito")\
    .select("codmes", "codclaveunicocli","codinternocomputacional", "codclavepartycli")\
        .filter(
            F.trim(F.col("codproductonivel0rbm")).isin('PYME REVOLVENTE','PYME NO REVOLVENTE')
            & (F.col("codmes") >= 202405)
            & (F.col("codclaveunicocli").isNotNull())
            & (F.col("ctdmesmaduracion") > 0)
            )\
            .distinct()
            
bd_202509_202601 = sp_clientes0\
    .groupBy(
        "codclaveunicocli", "codmes"
        ).agg(
            F.max(F.col("codinternocomputacional")).alias("codinternocomputacional"),
            F.max(F.col("codclavepartycli")).alias("codclavepartycli"),
            )

bd_202509_202601.count()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType
from pyspark.sql.functions import udf, array

# codmes_data = 202512 # un mes menos del codmes proceso
# codmes_proceso = 202601

# ============================================================
# BASE: Universo de clientes
# ============================================================

bd_202509_202601.cache()
print(f"Registros base: {bd_202509_202601.count()}")

# ============================================================
# PASO 1: tippartyidentificacion
# ============================================================

cliente_prospecto = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_modelogestion_vu.hm_clienteprospectopyme"
).filter(F.col("codmes") >= 202404)
 .select(
    F.col("CODMES").alias("CODMES_0"),
    "COCLAVEUNICOCLI",
    "tippartyidentificacion"
)

cliente_prospecto = cliente_prospecto.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
cliente_prospecto = cliente_prospecto.drop("CODMES_0")

# ================================
# PASO 2: Relacionados
# ================================

relacionados = spark.table(
    "catalog_lhcl_prod_bcp_bdv_rbmbcapym_apppyme_vu.hm_relacionadoapppyme"
).filter(f.col("codmes") >= 202404)\
 .select(
    f.col("CODMES").alias("CODMES_0"),
    "CODCLAVEUNICOCIL",
    "CODCLAVEUNICOCILREL",
    "DESTIPREL",
    "PCTPARTICIPACIONREL"
)

relacionados = relacionados.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
relacionados = relacionados.drop("CODMES_0")

relacionados = relacionados.join(
    cliente_prospecto,
    ["codmes", "CODCLAVEUNICOCIL"],
    "left_outer"
)

# ================================
# PASO 3: Filtrar DUENIO/DUENIO P.N.
# ================================

duenios = relacionados\
 .filter(f.col("DESTIPREL").isin('DUENIO', 'DUENIO P.N.'))\
 .select(
    "codmes",
    "CODCLAVEUNICOCIL",
    "CODCLAVEUNICOCILREL",
    f.when(f.col("tippartyidentificacion") == '6', 'J')
     .otherwise('N').alias("FLGTIPPER")
)

# ================================
# PASO 4: UN SOLO dueño por cliente
# ================================

duenio_unico = duenios\
    .groupBy("codmes","CODCLAVEUNICOCLI")\
    .agg(
        F.max(
            F.when(F.col('FLGTIPPER') == 'N', F.col('CODCLAVEUNICOCLI'))\
            .otherwise(F.col('CODCLAVEUNICOCLIREL'))\
        ).alias('CODCLAVEUNICOCLIREL')
    )

# ================================
# PASO 5: Marcar FLGRLDUENIO
# ================================

relacionados_con_flag = relacionados.alias("A").join(
    duenio_unico.alias("B"),
    (F.col("A.CODCLAVEUNICOCLI") == F.col("B.CODCLAVEUNICOCLI")) &
    (F.col("A.CODCLAVEUNICOCLIREL") == F.col("B.CODCLAVEUNICOCLIREL")) &
    (F.col("A.codmes") == F.col("B.codmes")),
    "left_outer"
).select(
    F.col("A.codmes"),
    F.col("A.CODCLAVEUNICOCLI"),
    F.col("A.CODCLAVEUNICOCLIREL"),
    F.col("A.DESTIPREL"),
    F.col("A.PCTPARTICIPACIONREL"),
    F.when(
        F.col("A.DESTIPREL").isin('DUENIO', 'DUENIO_P_N'),
        F.when(F.col("B.CODCLAVEUNICOCLI").isNull(), 0).otherwise(1)
    ).otherwise(0).alias("FLGRLDUENIO")
)
# ================================
# PASO 6: Filtrar solo el dueño seleccionado
# ================================
relacion_dueno_final = relacionados_con_flag\
    .filter(F.col('FLGRELDUENIO') == 1)\
    .select('codmes', 'CODCLAVEUNICOCIL', 'CODCLAVEUNICOCILREL')

# ================================
# PASO 7: Campos del UNIVERSO (edad_fin, act_eco_fin)
# ================================
universo_apppyme = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_clientepreaprobacionapppyme"
).filter(F.col("codmes") >= 202404)\
 .select(
    F.col("CODMES").alias("CODMES_0"),
    "CODCLAVEUNICOCIL",
    F.col("NUMEDAD").cast(IntegerType()).alias("EDAD_FIN"),
    F.col("DESSECCIONECONOMICAAPPPYME").alias("ACT_ECO_FIN"),
    "TIPPARTYIDENTIFICACION"
)

universo_apppyme = universo_apppyme.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
universo_apppyme = universo_apppyme.drop('CODMES_0')

# ================================
# PASO 8: Variables de CARRETERA (hm_matrizdeudorrccproducto)
# ================================
carretera_rcc = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
).filter(F.col("codmes") >= 202404)\
 .select(
    F.col("CODMES").alias("CODMES_0"),
    "codclaveunicocl",
    F.col("RCC_TIP_COND_MOR_MAX_CRNNR_MAX_U12").alias("ATRASOMAX_CRNNR_12"),
    F.col("RCC_MTO_ADE_ACT_SHIP_RT_UGM").alias("MONTOADE_ACT_MAX6_S_HIP"),
    F.col("RCC_PCT_UTL_3_RT_U3M").alias("UTL_3"),
    F.col("RCC_CTD_SF_CAL_CPP_FRQ_0_U24").alias("SF_NUM_CAL_CPP24")
)

carretera_rcc = carretera_rcc.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
carretera_rcc = carretera_rcc.drop('CODMES_0')

# ============================================
# PASO 9: Variables de RESUMEN SALDO (hm_matrizresumensaldoactivopasivo)
# ============================================

resumen_saldo = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo"
).filter(F.col("codmes") >= 202404)\
 .select(
    F.col("CODMES").alias("CODMES_0"),
    codclaveunicolci,
    F.col("PROD_MTO_SLD_PRM_TSAV_FRO_100_U24").alias("MESES_PHSAVMF_24_100"),
    F.col("PROD_CTD_MES_PAS_ACT_MAX_V2_1000_U12").alias("MESES_PASIVO_ACTIVO_12_1000")
)

resumen_saldo = resumen_saldo.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
resumen_saldo = resumen_saldo.drop('CODMES_0')

# ============================================
# PASO 10: Variables de MATERIALIDAD
# ============================================

materialidad = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablerccmaterialidadapppyme"
).filter(F.col("codmes") >= 202404)\
 .select(
    F.col("CODMES").alias("CODMES_0"),
    codclaveunicolci,
    F.col("RCC_TIP_COND_MOR_MAX_CRNNR_100_MAX_U12").alias("ATRASOMAX_CRNNER_12_100"),
    F.col("RCC_TIP_CF_CAL_CPP_FRO_100_U24").alias("SF_NUM_CAL_CPP24_100"),
    F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_100_UGM").alias("MESES_ACTIVO_SF_BU_MAG_100")
)

materialidad = materialidad.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
materialidad = materialidad.drop('CODMES_0')

# ============================================
# PASO 11: Construir carretera CON materialidad (fullouter)
# ============================================

# Primero unir carretera_rcc + resumen_saldo
carretera_completa = carretera_rcc.join(
    resumen_saldo,
    ["codmes","codclaveunicoli"],
    "fullouter")

# Luego unir con materialidad
carretera_con_mat = carretera_completa.join(
    materialidad,
    ["codmes","codclaveunicoli"],
    "fullouter")

# Aplicar lógica de materialidad: si existe _100, se usa; si no, se usa el base
carretera_final = carretera_con_mat.select(
    "codmes",
    "codclaveunicoli",
    # ATRASOMAX_CRNENR_12: materialidad reemplaza
    F.when(F.col("ATRASOMAX_CRNENR_12_100").isNull(), F.col("ATRASOMAX_CRNENR_12"))
     .otherwise(F.col("ATRASOMAX_CRNENR_12_100")).alias("ATRASOMAX_CRNENR_12"),
    # SF_NUM_CAL_CPP24: materialidad reemplaza
    F.when(F.col("SF_NUM_CAL_CPP24_100").isNull(), F.col("SF_NUM_CAL_CPP24"))
     .otherwise(F.col("SF_NUM_CAL_CPP24_100")).alias("SF_NUM_CAL_CPP24"),
    # MESES_ACTIVO_SF_BU_MA6_0: materialidad reemplaza
    # OJO: en el código original el campo base es RCC_CTD_MES_ACT_SF_BUEN_MAL_0_UGM
    # pero en variablesMaterialidad se mapea MESES_ACTIVO_SF_BU_MA6_100 ->
    F.col("MESES_ACTIVO_SF_BU_MA6_100").alias("MESES_ACTIVO_SF_BU_MA6_0_MAT"),
    # Campos directos (sin materialidad)
    "MONTOADE_ACT_MA6_S_HIP",
    "UTL_3",
    "MESES_PMSAVMF_24_100"
)
# ================================================
# PASO 11b: Necesitamos el campo base MESES_ACTIVO_SF_BU_MA6_0 de carretera
# ================================================

carretera_meses_sf = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
).filter(f.col("codmes") >= 202404)
.select(
    F.col("CODMES").alias("CODMES_0"),
    "codclaveunicocli",
    F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_0_UGM").alias("MESES_ACTIVO_SF_BU_MA6_0_BASE")
)

carretera_meses_sf = carretera_meses_sf.withColumn('codmes', add_codmes_spark("CODMES_0", +1))
carretera_meses_sf = carretera_meses_sf.drop("CODMES_0")

carretera_final = carretera_final.join(carretera_meses_sf, ["codmes","codclaveunicocli"], "left_outer")

# Aplicar materialidad para MESES_ACTIVO_SF_BU_MA6_0
carretera_final = carretera_final.withColumn(
    "MESES_ACTIVO_SF_BU_MA6_0",
    F.when(f.col("MESES_ACTIVO_SF_BU_MA6_0_MAT").isNull(), F.col("MESES_ACTIVO_SF_BU_MA6_0_BASE"))
    .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_MAT"))
).drop("MESES_ACTIVO_SF_BU_MA6_0_MAT", "MESES_ACTIVO_SF_BU_MA6_0_BASE")

# ==========================================
# PASO 12: Obtener variables del DUEÑO (_RL)
# ==========================================

variables_dueno = relacion_dueno_final.join(
    carretera_final.select(
        F.col("codmes"),
        F.col("codclaveunicocli").alias("CODCLAVEUNICOCLIREL"),
        F.col("ATRASOMAX_CRNNER_12").alias("ATRASOMAX_CRNNER_12_RL"),
        F.col("MONTOADE_ACT_MAX6_S_HIP").alias("MONTOADE_ACT_MAX6_S_HIP_RL"),
        F.col("UTL_3").alias("UTL_3_RL"),
        F.col("SF_NUM_CAL_CPP24").alias("SF_NUM_CAL_CPP24_RL"),
        F.col("MESES_ACTIVO_SF_BU_MA6_0").alias("MESES_ACTIVO_SF_BU_MA6_0_RL"),
        F.col("MESES_PMSAVMF_24_100").alias("MESES_PMSAVMF_24_100_RL")
    ),
    ["CODCLAVEUNICOCLIREL", "codmes"],
    "left_outer"
).select(
    "codmes",
    "CODCLAVEUNICOCLI",
    "ATRASOMAX_CRNNER_12_RL",
    "MONTOADE_ACT_MAX6_S_HIP_RL",
    "UTL_3_RL",
    "SF_NUM_CAL_CPP24_RL",
    "MESES_ACTIVO_SF_BU_MA6_0_RL",
    "MESES_PMSAVMF_24_100_RL"
)

# Verificar unicidad
variables_dueno.groupBy("codmes", "CODCLAVEUNICOCLI").count() \
    .filter(F.col("count") > 1).show()

# ============================================
# PASO 13: Obtener variables del CLIENTE (para AG)
# ============================================
# SF_NUM_CAL_CPP24 y MESES_ACTIVO_SF_BU_MAG_0 del propio cliente
variables_cliente = carretera_final.select(
    "codes",
    "codclaveunicocl1",
    "SF_NUM_CAL_CPP24",
    "MESES_ACTIVO_SF_BU_MAG_0"
)

# ============================================
# PASO 14: UDF para calcular MAX entre columnas (replica del código original)
# ============================================
def operacionesMaxBetweenCols(columnas):
    resultado = 0.0
    valorInicial = float(-9999999999999999.0)
    mayor = float(0.0)
    numero = float(0.0)
    for column in columnas:
        if column is not None and str(column) != "" and str(column).upper() != "NULL":
            numero = float(column)
        else:
            numero = -9999999999999999.0
        if numero >= valorInicial:
            mayor = float(numero)
            valorInicial = float(numero)
        elif numero < valorInicial:
            mayor = float(valorInicial)
        if mayor == -9999999999999999.0:
            resultado = None
        else:
            resultado = mayor
    return resultado

operacionesMaxBetweenCols_udf = udf(operacionesMaxBetweenCols, DoubleType())

# ================================
# PASO 15: Unir todo y calcular campos AG
# ================================

resultado = bd_202509_202601 \
    .join(universo_appynae, ["codmes","CODCLAVEUNICOCLI"], "left_outer") \
    .join(variables_dueno, ["codmes","CODCLAVEUNICOCLI"], "left_outer") \
    .join(variables_cliente, ["codmes","CODCLAVEUNICOCLI"], "left_outer")

# --- SF_NUM_CAL_CPP24_AG ---
# Si PN (tippartyidentificacion != '6'): valor del cliente
# Si PJ (tippartyidentificacion == '6'): max(cliente, dueño)
resultado = resultado.withColumn(
    "SF_NUM_CAL_CPP24_AG_raw",
    operacionesMaxBetweenCols_udf(
        array(F.col("SF_NUM_CAL_CPP24"), F.col("SF_NUM_CAL_CPP24_RL"))
    )
).withColumn(
    "SF_NUM_CAL_CPP24_AG",
    F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("SF_NUM_CAL_CPP24"))
     .otherwise(F.col("SF_NUM_CAL_CPP24_AG_raw"))
     .cast(IntegerType())
)

# --- MESES_ACTIVO_SF_BU_MAG_0_AG ---
# Misma lógica: PN -> cliente, PJ -> max(cliente, dueño)
resultado = resultado.withColumn(
    "MESES_ACTIVO_SF_BU_MAG_0_AG_raw",
    operacionesMaxBetweenCols_udf(
        array(F.col("MESES_ACTIVO_SF_BU_MAG_0"), F.col("MESES_ACTIVO_SF_BU_MAG_0_RL"))
    )
).withColumn(
    "MESES_ACTIVO_SF_BU_MAG_0_AG",
    F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("MESES_ACTIVO_SF_BU_MAG_0"))
     .otherwise(F.col("MESES_ACTIVO_SF_BU_MAG_0_AG_raw"))
     .cast(IntegerType())
)


# ============================================
# PASO 16: Castear campos _RL a tipos correctos
# ============================================

resultado_bd_APP_SCORE_APROB_PYME = resultado.select(
    "codmes",
    "codclaveunicoli",
    "codclavepartycli",
    "codinternocomputacional",
    # Campos del universo
    F.col("EDAD_FIN").cast(IntegerType()).alias("edad_fin"),
    F.col("ACT_ECO_FIN").alias("act_eco_fin"),
    # Campos _RL (del dueño)
    F.col("ATRASOMAX_CRNENR_12_RL").cast(IntegerType()).alias("atrasomax_crnenr_12_rl"),
    F.col("MONTOADE_ACT_MAX6_S_HIP_RL").cast(DecimalType(precision=19, scale=8)).alias("montoade_act_max6_s_hip_rl"),
    F.col("UTL_3_RL").cast(DecimalType(precision=23, scale=6)).alias("utl_3_rl"),
    F.col("MESES_PMSAVMF_24_100_RL").cast(IntegerType()).alias("meses_pmsavmf_24_100_rl"),
    # Campos _AG (agregados)
    F.col("SF_NUM_CAL_CPP24_AG").cast(IntegerType()).alias("sf_num_cal_cpp24_ag"),
    F.col("MESES_ACTIVO_SF_BU_MA6_0_AG").cast(IntegerType()).alias("meses_activo_sf_bu_ma6_0_ag")
)



##VARIABLES de la tabla "hm_scoreappbasepymemodulodemografico"

In [0]:

# ===============================================================
# VARIABLES CON UN MES DE DESFASE:
# las siguientes variables ya fueron construidas con información de CODMES_DATA 202505
# para ser utilizadas en la actualización de la tabla BHV correspondiente al
# CODMES_DATA = 202506. Es decir, estas variables tienen un desfase de 1 mes
# respecto al periodo de actualización.
# ===============================================================


###RECABLEO DE VARIABLES INPUT

In [0]:
# bd_MOD_DEMO = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemodulodemografico") \
#    .select(F.col("CODMES").alias("CODMES_0"), "codeclaveunicocl", 
#            F.col("ctdmesantiguedadempsunat").alias("MOD_DEMO__ctdmesantiguedadempsunat")) \
#    .filter(F.col("codmes") == 202505) \
#    .distinct()
# bd_MOD_DEMO = bd_MOD_DEMO.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
# bd_MOD_DEMO = bd_MOD_DEMO.drop("CODMES_0")

In [0]:
## ctdmesantiguedadempsunat

import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# ================================================
# PASO 1: Obtener ctdmesantiguedadempsunat
# -----------------------------------------------
contribuyente_sunat = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmccapym_modelogestion_vu.hm_contribuyentesunatpyme"
).select(
    F.col("CODMES").alias("CODMES_0"),
    F.col("CODCLAVEUNICOCLI"),
    F.col("CTDMESANTIGUEDADEMP").cast(IntegerType()).alias("ctdmesantiguedadempsunat")
)

# Desfase: la data es un mes antes del codmes_proceso
contribuyente_sunat = contribuyente_sunat.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
contribuyente_sunat = contribuyente_sunat.drop("CODMES_0")

# ================================================
# PASO 2: Unir al universo principal
# ================================================
resultado_bd_MOD_DEMO = bd_202509_202601.join(
    contribuyente_sunat,
    ["CODCLAVEUNICOCLI", "codmes"],
    "left_outer"
)


##Variables de la tabla “hm_scoreappbasepymemoduloactivo”

In [0]:
# VARIABLES CON UN MES DE DESFASE:
# Las siguientes variables ya fueron construidas con información de CODMES_DATA 202505
# para ser utilizadas en la actualización de la tabla BHW correspondiente al
# CODMES_DATA = 202506. Es decir, estas variables tienen un desfase de 1 mes
# respecto al periodo de actualización.


###RECABLEO DE VARIABLES INPUT

In [0]:
# bd_MOD_ACT = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepymemoduolactivo")
#   .select(f.col("CODMES_0").alias("CODMES_0"), "codclaveunicolli",
#           f.col("pctratiomotdeceduapymermtopasivoprmu3m").alias("MOD_ACT_pctratiomotdeceduapymermtopasivoprmu3m"),
#           f.col("pctratiomotcoopeaprmu6mopecprmu12").alias("MOD_ACT_pctratiomotcoopeaprmu6mopecprmu12"),
#           f.col("pctratiomotldcapitalmtosldviggasprmu6m").alias("MOD_ACT_pctratiomotldcapitalmtosldviggasprmu6m"),
#           f.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12").alias("MOD_ACT_isav_mto_opea_estvta_pym_u6m_rt_max_u12"))
#   .filter(f.col("codmes") == 202505)
#   .distinct()
# bd_MOD_ACT = bd_MOD_ACT.withColumn("codmes", add_codmes_spark("CODMES_0", +1))
# bd_MOD_ACT = bd_MOD_ACT.drop("CODMES_0")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.window import Window
from dateutil.relativedelta import relativedelta
from datetime import datetime

# ==================================================
# PARAMETROS
# ==================================================

codmescuatro = 202502
codmestres = 202503
codmes_data_1 = 202505
codmes_data = 202506

def generar_meses(mes_inicio, mes_fin):
    meses = []
    fecha_fin = datetime.strptime(str(mes_fin), "%Y%m")
    fecha_actual = datetime.strptime(str(mes_inicio), "%Y%m")
    while fecha_actual <= fecha_fin:
        meses.append(fecha_actual.strftime("%Y%m"))
        fecha_actual += relativedelta(months=1)
    return meses

# ================================
# UNIVERSO MÓDULO ACTIVO (filtro clave)
# Solo clientes con flgactmay100may6u12 = 1
# ================================

universo_segmento = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_segmentacionpyme_vu.hm_segmentoinformativopyme"
).filter(
    (F.col("codmes") == codmes_data_1) &
    (F.col("flgactmay100may6u12") == 1)
).select("CODCLAVEUNICCOLI").distinct()

# ================================
# CAMPO 1 y 3: Transacciones
# ================================

transacciones = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_matrizbasepasivoclienteapppyme"
).filter(F.col("codmes") == codmes_data_1)
.select(
    F.col("CODCLAVEPARTYCLI"),
    (F.col("ISAV_MTO_OPEA_ESTVTA_PYM_PRM_U6M") / F.col("ISAV_MTO_OPEA_ESTVTA_PYM_MAX_U12"))
        .alias("isav_mto_opea_estvta_pym_u6m_rt_max_u12"),
    (F.col("ISAV_MTO_OPEA_ESTVTA_PYM_U6M") / F.col("ISAV_MTO_OPEC_ESTVTA_PYM_PRM_U12"))
        .alias("pctratmtoopeapermu6mopecprmu12")
)

# ==================================================
# CAMPO 2: Variación de deuda / Pasivos
# ==================================================

# Portafolio
portafolio = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito"
).filter(
    (F.col("CODMES").between(codmescuatro, codmes_data_1)) &
    (F.trim(F.col("FLGCTAVALIDA")) == "1") &
    (F.col("TIPESTADOCTA").isin("A", "AC", "D"))
).select("CODCLAVEPARTYCLI", "MTOSALDOCAPITALSOL", "CODMES")

portafolio_agrupado = portafolio.groupBy("CODCLAVEPARTYCLI", "CODMES").agg(
    F.sum("MTOSALDOCAPITALSOL").alias("mtosaldocapitalsol_tmp")
)

# CrossJoin con todos los meses
lista_meses = generar_meses(codmescuatro, codmes_data_1)
df_meses = spark.createDataFrame(lista_meses, StringType()).select(F.col("value").alias("CODMES"))

partys = portafolio_agrupado.select("CODCLAVEPARTYCLI").dropDuplicates()
cross = partys.crossJoin(F.broadcast(df_meses))
portafolio_completo = portafolio_agrupado.join(cross, on=["CODMES", "CODCLAVEPARTYCLI"], how="right_outer")

# Variación
w = Window.partitionBy("CODCLAVEPARTYCLI").orderBy(F.col("CODMES").desc())

portafolio_var = portafolio_completo.withColumn(
    "mtosaldocapitalsol_tmp_prev",
    F.lead("mtosaldocapitalsol_tmp", 1).over(w).cast("double")
).withColumn(
    "mtosaldocapitalsol_tmp_VAR_DEC",
    F.when(
        F.col("mtosaldocapitalsol_tmp").isNull() & F.col("mtosaldocapitalsol_tmp_prev").isNull(),
        F.lit(None)
    ).otherwise(
        F.when(
            F.coalesce(F.col("mtosaldocapitalsol_tmp"), F.lit(0)) -
            F.coalesce(F.col("mtosaldocapitalsol_tmp_prev"), F.lit(0)) > 0,
            F.lit(0)
        ).otherwise(
            F.abs(
                F.coalesce(F.col("mtosaldocapitalsol_tmp"), F.lit(0)) -
                F.coalesce(F.col("mtosaldocapitalsol_tmp_prev"), F.lit(0))
            )
        )
    )
)

dec_var_prm3 = portafolio_var.filter(
    F.col("CODMES").between(str(codmes_tres), str(codmes_data_1))
).groupBy("CODCLAVEPARTYCLI").agg(
    F.avg("mtosaldocapitalsol_tmp_VAR_DEC").alias("dec_var_MONTO_meanPrev3")
)

# Pasivos
pasivos = spark.table(
    "catalog_lhcl_prod_bcp.bcp_ddv_rbmcbapym_apppyme_vu.hm_matrizbaseresumensaldoapppyme"
).filter(F.col("codmes") == codmes_data_1)\
.select("CODCLAVEUNICCOLI", "PROD_MTO_SLD_PRM_VIG_PAS_AHO_CTEORD_PRM_U3M")

# ================================
# UNIR TODO CON FILTRO DE UNIVERSO DEL CODMES = CODMES_DATA
# ================================

# Marcar quiénes están en el universo del módulo activo
bd_202509_202601_f = bd_202509_202601.filter(
    F.col("CODMES") == codmes_data
)

bd_con_flag = bd_202509_202601_f.join(
    universo_segmento.withColumn("en_universo", F.lit(1)),
    "CODCLAVEUNICCOLI",
    "left"
)

# Joins de insumos
resultado = bd_con_flag\
    .join(transacciones, "CODCLAVEPARTYCLI", "left")\
    .join(dec_var_prm3, "CODCLAVEPARTYCLI", "left")\
    .join(pasivos, "CODCLAVEUNICCOLI", "left")

# Calcular campo 2 intermedio
resultado = resultado.withColumn(
    "pctratomotdeceduadepymrtoptpasivoprmu3m_raw",
    F.col("dec_var_MONTO_meanPrev3") / F.col("PROD_MTO_SLD_PRM_VIG_PAS_AHO_CTEORD_PRM_U3M")
)

# ================================
# APLICAR REGLA: Si NO está en universo -> NULL
# ================================

resultado = resultado.withColumn(
    "isav_mto_opea_estvta_pym_u6m_rt_max_u12",
    F.when(F.col("en_universo") == 1, F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12"))
     .otherwise(F.lit(None))
).withColumn(
    "pctratiomtodecdeudapymermtopasivoprmu3m",
    F.when(F.col("en_universo") == 1, F.col("pctratiomtodecdeudapymermtopasivoprmu3m_raw"))
     .otherwise(F.lit(None))
).withColumn(
    "pctratiomtoopeaprmu6mopepcrpmu12",
    F.when(F.col("en_universo") == 1, F.col("pctratiomtoopeaprmu6mopepcrpmu12"))
     .otherwise(F.lit(None))
)

# ================================
# Selección final con cast
# ================================
resultado_bd_MOD_ACT = resultado.select(
    "codmes",
    "codclavenunicli",
    F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12").cast(DecimalType(19, 8)).alias("isav_mto_opea_estvta_pym_u6m_rt_max_u12"),
    F.col("pctratiomtodecdeudapymermtopasivoprmu3m").cast(DecimalType(19, 8)).alias("pctratiomtodecdeudapymermtopasivoprmu3m"),
    F.col("pctratiomtoopeaprmu6mopepcrpmu12").cast(DecimalType(19, 8)).alias("pctratiomtoopeaprmu6mopepcrpmu12")
)

#UNIVERSO PARA EL CODMES DATA 202506

In [0]:

clientes0 = spark.table("catalog_hlcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito")\
    .select("codmes", "codclaveunicocli", "codinternocomputacional", "codclavepartycli")\
        .filter(
            F.trim(F.col("codproductonivel0rbm")).isin('PYME REVOLVENTE','PYME NO REVOLVENTE')
            & (F.col("codmes") == 202506)
            & (F.col("codclaveunicocli").isNotNull())
            #& (F.col("codinternocomputacional").isNotNull())
            & (F.col("ctdmesmaduracion") > 0)
                )\
                .distinct()

In [0]:
clientes = clientes0\
    .groupBy(
        "codclaveunicocli", "codmes"
    )\
    .agg(
        F.max(F.col("codinternocomputacional")).alias("codinternocomputacional"),
        F.max(F.col("codclavepartycli")).alias("codclavepartycli"),
    )

clientes.count() #319155

#CONSTRUCCION DE VARIABLES DEL MODELO


##VARIABLES QUE YA SE HAN CONSTRUIDO CON UN MES DE DESFASE, SE HAN CONSTRUIDO CON CODMES = 202505, SON VARIABLES DE TABLAS SCORE

In [0]:
# bd_MOD_DEMO = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scoreappbasepyemodulodemografico")\
#     .select(f.col("CODMES").alias("CODMES_0"), "codclaveunicoli",
#             f.col("ctdmesantiguedadempsunat").alias("MOD_DEMO__ctdmesantiguedadempsunat"))\
#     .filter(f.col("codmes") == 202505)\
#     .distinct()
# bd_MOD_DEMO = bd_MOD_DEMO.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
# bd_MOD_DEMO = bd_MOD_DEMO.drop("CODMES_0")

bd_MOD_DEMO = resultado_bd_MOD_DEMO\
    .select("CODMES",
            "codclaveunicoli",
            f.col("ctdmesantiguedadempsunat").alias("MOD_DEMO__ctdmesantiguedadempsunat"))\
    .filter(f.col("codmes") == 202506)\
    .distinct()

#bd_MOD_ACT = spark.table("catalog_lhc1_prod_bcp.#bcp_ddv_rmbcapym_apppyme_vu_hm_scoreappbasepymeoduloactivo")\
#    .select(F.col("CODMES").alias("CODMES_0"), "codclaveuniccoli",
#        F.col("pctratiomtodeduedupaymermtopasivoprmu3m").alias("MOD_ACT_pctratiomtodeduedupaymermtopasivoprmu3m"),
#        F.col("pctratiomtoopeaprmu6mopecrpmu12").alias("MOD_ACT_pctratiomtoopeaprmu6mopecrpmu12"),
#        F.col("pctratiomtolsldcapitalmtosldvigpasprmu6m").alias("MOD_ACT_pctratiomtolsldcapitalmtosldvigpasprmu6m"),
#        F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12").alias("MOD_ACT_isav_mto_opea_estvta_pym_u6m_rt_max_u12")
#    )\
#    .filter(F.col("codmes") == 202505)\
#    .distinct()

bd_MOD_ACT = resultado_bd_MOD_ACT\
    .select("CODMES",
        "codclaveuniccoli",
        F.col("pctratiomtodeduedupaymermtopasivoprmu3m").alias("MOD_ACT_pctratiomtodeduedupaymermtopasivoprmu3m"),
        F.col("pctratiomtoopeaprmu6mopecrpmu12").alias("MOD_ACT_pctratiomtoopeaprmu6mopecrpmu12"),
        F.col("isav_mto_opea_estvta_pym_u6m_rt_max_u12").alias("MOD_ACT_isav_mto_opea_estvta_pym_u6m_rt_max_u12")
    )\
    .filter(F.col("codmes") == 202506)\
    .distinct()

# bd_APP_SCORE_APROB_PYME = spark.table("catalog_lhc1_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_scorepreaprobacionapppyme")\
#     .select(F.col("CODMES").alias("CODMES_0"), "codclaveunicocli",
#             F.col("atrasomax_crnenr_12_rl").alias("APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl"),
#             F.col("meses_pmsavmf_24_100_rl").alias("APP_SCORE_APROB_PYME__meses_pmsavmf_24_100_rl"),
#             F.col("montoade_act_max6_s_hip_rl").alias("APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl"),
#             F.col("sf_num_cal_cpp24_ag").alias("APP_SCORE_APROB_PYME__sf_num_cal_cpp24_ag"),
#             F.col("utl_3_rl").alias("APP_SCORE_APROB_PYME__utl_3_rl"),
#             F.col("act_eco_fin").alias("APP_SCORE_APROB_PYME__act_eco_fin"),
#             F.col("edad_fin").alias("APP_SCORE_APROB_PYME__edad_fin"),
#             F.col("meses_activo_sf_bu_ma6_0_ag").alias("APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag"),
#             )\
#     .filter(F.col("codmes") == 202505)\
#     .distinct()
# bd_APP_SCORE_APROB_PYME = bd_APP_SCORE_APROB_PYME.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
# bd_APP_SCORE_APROB_PYME = bd_APP_SCORE_APROB_PYME.drop("CODMES_0")

bd_APP_SCORE_APROB_PYME = resultado_bd_APP_SCORE_APROB_PYME\
    .select("CODMES",
        "codclaveunicocl1",
        F.col("atrasomax_crnner_12_rl").alias("APP_SCORE_APROB_PYME_atrasomax_crnner_12_rl"), #####
        F.col("meses_pmsavmf_24_100_rl").alias("APP_SCORE_APROB_PYME_meses_pmsavmf_24_100_rl"), #####
        F.col("montoade_act_max6_s_hip_rl").alias("APP_SCORE_APROB_PYME_montoade_act_max6_s_hip_rl"), #####
        F.col("sf_num_cal_cpp24_ag").alias("APP_SCORE_APROB_PYME_sf_num_cal_cpp24_ag"), #####
        F.col("utl_3_rl").alias("APP_SCORE_APROB_PYME_utl_3_rl"), #####
        F.col("act_eco_fin").alias("APP_SCORE_APROB_PYME_act_eco_fin"), #####
        F.col("edad_fin").alias("APP_SCORE_APROB_PYME_edad_fin"), #####
        F.col("meses_activo_sf_bu_ma6_0_ag").alias("APP_SCORE_APROB_PYME_meses_activo_sf_bu_ma6_0_ag"), #####
    )\
    .filter(F.col("codmes") == 202506)\
    .distinct()

bd_APP_SCORE_APROB_PYME = bd_APP_SCORE_APROB_PYME.withColumn("RNG_ACTIVIDAD_ECONOM",
    F.when(F.col("APP_SCORE_APROB_PYME_act_eco_fin").isNull(), 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'PESCA', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'OTROS', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'SERVICIOS', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'ENERGIA', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'CONSTRUCCION', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'ADM_PUBLICA', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'ACT_INMOB_EMP_Y_DE_ALQ', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'INDUST_MANUFACT', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'COMERCIO', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'HOGAR', 1)
    .when(F.col("APP_SCORE_APROB_PYME_act_eco_fin") == 'SALUD', 1)
    .otherwise(0)
)

###VARIABLES CON DESFASE DE 1 MES

In [0]:
CONSOL_CALCUL_DEUD_RELAT = spark.table("catalog_lhcl_prod_bcp.
bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_consolidadovariabledeudavalorrelativizado")\
    .select(F.col("CODMES").alias("CODMES_0"),"codclaveunicocli",
            F.col("pctrelprmtoprisolu12m5").alias
            ("CONSOL_CALCUL_DEUD_RELAT_pctrelprmtoprisolu12m5"))\
    .filter(F.col("codmes") == 202505)\
    .distinct()

CONSOL_CALCUL_DEUD_RELAT = CONSOL_CALCUL_DEUD_RELAT.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
CONSOL_CALCUL_DEUD_RELAT = CONSOL_CALCUL_DEUD_RELAT.drop("CODMES_0")

VIDEVAR_MTX_MORA_POND_CLI_MMGR = spark.table("catalog_lhcl_prod_bcp.
bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_matrizmoraponderadaclientemngr")\
    .select(F.col("CODMES").alias("CODMES_0"),"codclaveunicocli",
            F.col("mtodeudaclasifriesgofactorsdctosolu2").alias
            ("VIDEVAR_MTX_MORA_POND_CLI_MMGR_mtodeudaclasifriesgofactorsdctosolu12"))\
    .filter(F.col("codmes") == 202505)\
    .distinct()

VIDEVAR_MTX_MORA_POND_CLI_MMGR = VIDEVAR_MTX_MORA_POND_CLI_MMGR.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
VIDEVAR_MTX_MORA_POND_CLI_MMGR = VIDEVAR_MTX_MORA_POND_CLI_MMGR.drop("CODMES_0")

PASIVO_EVOL_SALD_PYM = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_admrmgmr_videavariablesmodelos_vu.hm_variablepasivoevolucionalsaldopyme")\
    .select(F.col("CODMES").alias("CODMES_0"), "codinternocomputacional",
        F.col("mtoprnvariacionmensualprmvigsolu24m").alias("PASIVO_EVOL_SALD_PYM__mtoprnvariacionmensualprmvigsolu24m"),
        F.col("mtoprnvariacionmensualprmvigsolu6m").alias("PASIVO_EVOL_SALD_PYM__mtoprnvariacionmensualprmvigsolu6m"),
        F.col("pctprmdecvariacionmensualvigsolu3mu6m").alias("PASIVO_EVOL_SALD_PYM__pctprmdecvariacionmensualvigsolu3mu6m"),
        F.col("pctprmincrvariacionmensualprmvigsolu3mu6m").alias("PASIVO_EVOL_SALD_PYM__pctprmincrvariacionmensualprmvigsolu3mu6m"),
        F.col("pctprmincrvariacionmensualvigsolu6mu12m").alias("PASIVO_EVOL_SALD_PYM__pctprmincrvariacionmensualvigsolu6mu12m"),
        F.col("mtovaricionmensualprmvigsolu6m").alias("PASIVO_EVOL_SALD_PYM__mtovaricionmensualprmvigsolu6m"),
        F.col("mtovaricionmensualprmvigsol").alias("PASIVO_EVOL_SALD_PYM__mtovaricionmensualprmvigsol")
    )\
    .filter(F.col("codmes") == 202505)\
    .distinct()

PASIVO_EVOL_SALD_PYM = PASIVO_EVOL_SALD_PYM.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
PASIVO_EVOL_SALD_PYM = PASIVO_EVOL_SALD_PYM.drop("CODMES_0")

EVOL_COMP_TRX_PYM = spark.table("catalog_lhcl_prod_bcp_ddv_adrmmgr_videavariablesmodelos_vu.hm_evolucioncomportamientotransaccionalpyme")\
    .select(f.col("CODMES").alias("CODMES_0"), "codinternocomputacional",
    f.col("mtomaxtotalnetorxu6m").alias("EVOL_COMP_TRX_PYM_mtomaxtotalnetorxu6m"),
    f.col("pctratiomtototalabonoprmu3mu6m"),
    "EVOL_COMP_TRX_PYM_pctratiomtototalabonoprmu3mu6m",
    f.col("pctratiomtototalabonoprmu6mu12m").alias("EVOL_COMP_TRX_PYM_pctratiomtototalabonoprmu6mu12m"),
    f.col("pctratiomnutotalcargoprmu6mu12m").alias("EVOL_COMP_TRX_PYM_pctratiomnutotalcargoprmu6mu12m"))\
    .filter(f.col("codmes") == 202505)\
    .distinct()

EVOL_COMP_TRX_PYM = EVOL_COMP_TRX_PYM.withColumn('codmes', add_codmes_spark('CODMES_0', +1))
EVOL_COMP_TRX_PYM = EVOL_COMP_TRX_PYM.drop("CODMES_0")

###TABLAS SIN DESFASE


In [0]:
MTX_RCC_OTRA_DEUDA = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccotradeuda")\
    .select("codmes", "codclaveunicocli",
        F.col("rcc_mto_rdv_max_u3m").alias("MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m")
    )\
    .filter(F.col("codmes") == 202506)\
    .distinct()

CLASI_EXPER_CLI = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adr.mmgr_videavariablesmodelos_vu.hm_clasificacionclientenivelexperienciapyme")\
    .select("codmes", "codclaveunicocli",
        F.col("ctdempleado").alias("CLASI_EXPER_CLI__ctdempleado")
    )\
    .filter(F.col("codmes") == 202506)\
    .distinct()

EVOL_CLI_PYM = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adr.mmgr_videavariablesmodelos_vu.hm_variableactivoevolucionclientepyme")\
    .select("codmes", "codinternocomputacional",
        F.col("ctddiamoramay0u6m").alias("EVOL_CLI_PYM__ctddiamoramay0u6m"),
        F.col("ctdmaxdiamorau6m").alias("EVOL_CLI_PYM__ctdmaxdiamorau6m")
    )\
    .filter(F.col("codmes") == 202506)\
    .distinct()

MTX_RESUMEN_SALDO = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldo") \
    .select("codmes", "codclaveunicocli",
        F.col("prod_antmin_cvi_prm_u12").alias("MTX_RESUMEN_SALDO__prod_antmin_cvi_prm_u12"),
        F.col("prod_ctd_pai_max_u6m").alias("MTX_RESUMEN_SALDO__prod_ctd_pai_max_u6m"),
        F.col("prod_ctd_sld_act_u1m").alias("MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m"),
        F.col("prod_mto_sld_pai_g6m").alias("MTX_RESUMEN_SALDO__prod_mto_sld_pai_g6m")
    ) \
    .filter(F.col("codmes") == 202506) \
    .distinct()

MTX_RESUMEN_ACT_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo")\
    .select("codmes", "codclaveunicocli",
        f.col("prod_ctd_mes_pas_act_max_v2_0_u12").alias("MTX_RESUMEN_ACT_PAS__prod_ctd_mes_pas_act_max_v2_0_u12"),
        f.col("prod_mto_sld_fim_pas_min_24_24_rt_u24").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24"),
        f.col("prod_mto_sld_fim_tsav_max_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12"),
        f.col("prod_pct_fm_pmpas_med_12_12_rt_u12").alias("MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_12_12_rt_u12"),
        f.col("prod_pct_fm_pmpas_med_24_24_rt_u24").alias("MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_24_24_rt_u24"),
        f.col("prod_pct_fm_pmpas_med_6_6_rt_u6").alias("MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_6_6_rt_u6"),
        f.col("prod_pct_fm_pmpas_med_3_3_rt_u3").alias("MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_3_3_rt_u3"),
        f.col("prod_pct_fm_pmpas_med_1_1_rt_u1").alias("MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_1_1_rt_u1")
    )\
    .filter(f.col("codmes") == 202506)\
    .distinct()

MTX_MOV_ABONO_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu")\
hm_matrizmovimientoabonopasivo()\
.select("codmes", "codclaveunicolcli",
    F.col("isav_flg_opea_nhab_ncts_dol_max_u12").alias("MTX_MOV_ABONO_PAS__isav_flg_opea_nhab_ncts_dol_max_u12"),
    F.col("isav_tkt_opea_trnf_dol_max_u3m").alias("MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m"),
    F.col("isav_tkt_opea_trnf_min_u9m").alias("MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_min_u9m"),
    )\
.filter(F.col("codmes") == 202506)\
.distinct()

MTX_MOV_CARGO_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu")\
hm_matrizmovientocargopasivo()\
.select("codmes", "codclaveunicolcli",
    F.col("isav_mto_opec_retr_g6m").alias("MTX_MOV_CARGO_PAS__isav_mto_opec_retr_g6m"),
    F.col("isav_tkt_opec_pago_srv_prm_u3m").alias("MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m"),
    )\
.filter(F.col("codmes") == 202506)\
.distinct()

MTX_MOV_PAS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.")\
hm_matrizmovimientopasivo()\
.select("codmes", "codclaveunicolcli",
    F.col("isav_tkt_opec_sav_min_u9m").alias("MTX_MOV_PAS__isav_tkt_opec_sav_min_u9m"),
    )\
.filter(F.col("codmes") == 202506)\
.distinct()

MTX_TRX_CANAL_PAGO_TRANSF = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanalpagotransferencia")\
.select("codmes", "codclaveunicoli",
    F.col("can_ctd_tmo_tot_pag_bcp_frq_u6m").alias("MTX_TRX_CANAL_PAGO_TRANSF_can_ctd_tmo_tot_pag_bcp_frq_u6m"),
    F.col("can_mto_tmo_tot_pag_bcp_prm_u6m").alias("MTX_TRX_CANAL_PAGO_TRANSF_can_mto_tmo_tot_pag_bcp_prm_u6m"),
    F.col("can_tkt_tmo_tot_pag_srv_g6m").alias("MTX_TRX_CANAL_PAGO_TRANSF_can_tkt_tmo_tot_pag_srv_g6m"),
    F.col("can_tkt_tmo_tot_pag_srv_sol_g6m").alias("MTX_TRX_CANAL_PAGO_TRANSF_can_tkt_tmo_tot_pag_srv_sol_g6m")
    )\
.filter(F.col("codmes") == 202506)\
.distinct()

MTX_TRX_POS = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccionpos")\
.select("codmes", "codclaveunicoli",
    F.col("pos_tkt_trx_com_sol_prm_p6m").alias("MTX_TRX_POS_pos_tkt_trx_com_sol_prm_p6m"),
    F.col("pos_tkt_trx_td_prm_u6m").alias("MTX_TRX_POS_pos_tkt_trx_td_prm_u6m")
    )\
.filter(F.col("codmes") == 202506)\
.distinct()

MTX_RCC_PROD = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeucrproduct")\
.select("codmes", "codclaveunicoli",
    F.col("rcc_ctd_mes_act_sf_buen_1000_frq_v_u12").alias("MTX_RCC_PROD_rcc_ctd_mes_act_sf_buen_1000_frq_v_u12"),
    F.col("rcc_pct_deu_cpp_max_u12").alias("MTX_RCC_PROD_rcc_pct_deu_cpp_max_u12"),
    F.col("rcc_pct_deu_defc_max_u6m").alias("MTX_RCC_PROD_rcc_pct_deu_defc_max_u6m"),
    F.col("rcc_tip_cond_mor_max_crnor_max_u6m").alias("MTX_RCC_PROD_rcc_tip_cond_mor_max_crnor_max_u6m")
    )\
.filter(F.col("codmes") == 202506)\
.distinct()

bd_MTX_TRX_CANAL = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matriztransaccioncanal")\
    .select("codmes", "codclaveunicoli",
        F.col("can_tkt_tmo_tot_ret_sol_max_u6m").alias("MTX_TRX_CANAL__can_tkt_tmo_tot_ret_sol_max_u6m"),
        F.col("can_tkt_tmo_tot_sol_min_u12").alias("MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12")
    )\
    .filter(F.col("codmes") == 202506)\
    .distinct()

In [0]:
base_princ = clientes

# Joins por codclaveunicocli
joins_codclave = [
    bd_MOD_DEMO,
    bd_MOD_ACT,
    bd_APP_SCORE_APROB_PYME,
    CONSOL_CALCUL_DEUD_RELAT,
    VIDEVAR_MTX_MORA_POND_CLI_MMGR,
    MTX_RCC_OTRA_DEUDA,
    CLASI_EXPER_CLI,
    MTX_RESUMEN_SALDO,
    MTX_RESUMEN_ACT_PAS,
    MTX_MOV_ABONO_PAS,
    MTX_MOV_CARGO_PAS,
    MTX_MOV_PAS,
    MTX_TRX_CANAL_PAGO_TRANSF,
    MTX_TRX_POS,
    MTX_RCC_PROD,
    bd_MTX_TRX_CANAL
]

# Joins por codinternocomputacional
joins_codinterno = [
    PASIVO_EVOL_SALD_PYM,
    EVOL_COMP_TRX_PYM,
    EVOL_CLI_PYM
]

# Paso 1: Joins por codclaveunicocli
result = base_princ
for tabla in joins_codclave:
    result = result.join(
        tabla.select('codmes', 'codclaveunicocli', *[c for c in tabla.columns if c not in ['codmes', 'codclaveunicocli']]),
        on=['codmes', 'codclaveunicocli'],
        how='left'
    )

# Paso 2: Joins por codinternocomputacional
for tabla in joins_codinterno:
    result = result.join(
        tabla.select('codmes', 'codinternocomputacional', *[c for c in tabla.columns if c not in ['codmes', 'codinternocomputacional']]),
        on=['codmes', 'codinternocomputacional'],
        how='left'
    )

base_princ_final = result
base_princ_final.count()  #319155

In [0]:
GLOB_DUMMY_LIST = [111111111, -111111111, 222222222, -222222222, 333333333, -333333333,
444444444, 555555555, 666666666, 777777777, -99, -999,
222222222.2222, -333333333.3333, -99, -333333333.3333, 444444.4444,
555555.5555, 666666.6666, 777777.7777, 111111.1111, -111111.1111, 222222.2222,
-222222.2222, 333333.3333, -333333.3333, None]

cols = base_princ_final.columns
base_princ_final = (base_princ_final.select(*[f.when(f.col(c).isin(GLOB_DUMMY_LIST), None)\
.otherwise(f.col(c)).alias(c) for c in cols]))

In [0]:
result = base_princ_final
conteo_filas = result.count()

print(f"El total de registros es: {conteo_filas}")

chunk_size = int((conteo_filas / 1 * 0.1)) ## 202306 - 202601
round_size = (10 ** int(np.log10(chunk_size)) // 2)
chunk_size = int(np.ceil(chunk_size / round_size) * round_size)

# Convertir a DataFrame de Pandas
result_codmes = result.select("codmes").distinct()

for codmes in result_codmes.collect():
    try:
        output_path = f'/Workspace/Users/sherylsalazar@bcp.com.pe/02_Proyectos/
        06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/
        bd_universo_pd_bhv_troncal_202506_Valid_Impl/gen_{codmes["codmes"]}'
        os.makedirs(output_path, exist_ok = True)
        print(f"Procesando el codmes: {codmes['codmes']}")
        df_codmes = result.filter(result.codmes == codmes['codmes'])
        pdfdf = spark_to_pandas(df_codmes, batch_size=150, sort_by=['codmes', 'codclaveunicocl1'], verbose=True)
        pd_write_in_chunks(
            df=pdfdf,
            directory_path=output_path,
            chunk_size=chunk_size,
            chunk_name='data_partition',
            file_format='parquet',
            verbose=True,
            index=False)
        print(f"Se ha terminado de procesar el codmes: {codmes['codmes']}")
    except Exception as e:
        print(f"Error al procesar el codmes: {codmes['codmes']} debido a {e}")


In [0]:
df = pd_read_chunks(
    directory_path='/Workspace/Users/sherylsalazar@bcp.com.pe/02_Proyectos/    06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/
    bd_universo_pd_bhv_troncal_202506_Valid_Impl', file_format='parquet',
    verbose=True).sort_values(by=["codmes", "codclaveunicocl1"], ascending=[True, True])


In [0]:
for col in df.columns:
    try:
        df[col] = df[col].astype(float)
    except:
        pass

print(df.dtypes)

#TRANSFORMACIONES

In [0]:
# 0.999
col12 = "MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m"
df[col12 + "_cap"] = df[col12].clip(upper=54058.633)

# 0.999
col33 = "CLASI_EXPER_CLT__ctdempleado"
df[col33 + "_cap"] = df[col33].clip(lower=0, upper=203.0)

# 0.995
col17 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12"
df[col17 + "_cap"] = df[col17].clip(upper=88040.516)

# 0.999
col35 = "EVOL_CLT_PYME__ctdmaxdiamorau6m"
df[col35 + "_cap"] = df[col35].clip(lower=0, upper=59.0)

# 0.999
col123 = "MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m"
df[col123 + "_cap"] = df[col123].clip(upper=13.0)

# 0.995
col151 = "MOD_DEMO__ctdmestantiguedadempsunat"
df[col151 + "_cap"] = df[col151].clip(upper=396.0)

# 0.995
col32 = "APP_SCORE_APROB_PYME__utl_3_rl"
df[col32 + "_cap"] = df[col32].clip(lower=0, upper=261.9096)

# 0.998
col14 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24"
df[col14 + "_cap"] = df[col14].clip(upper=0.8768666)

# 0.999
col25 = "MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12"
df[col25 + "_cap"] = df[col25].clip(upper= 20916.992)

# 0.998
col21 = "MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmaact_24_24_rt_u24"
df[col21 + "_cap"] = df[col21].clip(upper= 19.579279)

# 0.99
col18 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m"
df[col18 + "_cap"] = df[col18].clip(upper= 4771.6816)

# 0.99
col19 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m"
df[col19 + "_cap"] = df[col19].clip(upper= 4.0419364)

# 0.995
col38 = "MOD_ACT__pctratiomtodecdeudapymermtopasivoprmu3m"
df[col38 + "_cap"] = df[col38].clip(upper= 480.26755)

col30 = "APP_SCORE_APROB_PYME__edad_fin"
df[col30 + "_cap"] = df[col30].clip(lower=25, upper=75)

# 0.999
col17 = "PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m"
df[col17 + "_cap"] = df[col17].clip(upper= 211663.56)

# 0.998
col16 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m"
df[col16 + "_cap"] = df[col16].clip(upper= 386732.84)

# 0.999
col11 = "MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_min_u9m"
df[col11 + "_cap"] = df[col11].clip(upper= 46430.434)

# 0.999
col13 = "MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m"
df[col13 + "_cap"] = df[col13].clip(upper= 469.555)

# 0.999
col11 = "VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasfriesgofactorsdctosolu12"
df[col11 + "_cap"] = df[col11].clip(upper= 709515.0)

# 0.999
col37 = "MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12"
df[col37 + "_cap"] = df[col37].clip(lower=0, upper=1)

# 0.999
col15 = "MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12"
df[col15 + "_cap"] = df[col15].clip(upper= 372685.25)

# 0.999
col22 = "MTX_RESUMEN_SALDO__prod_antmin_cvi_prm_u12"
df[col22 + "_cap"] = df[col22].clip(upper= 318.5)

# 0.999
col24 = "MTX_RESUMEN_SALDO__prod_mto_sld_pai_g6m"
df[col24 + "_cap"] = df[col24].clip(upper= 58.04616)

# 0.999
col152 = "MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m"
df[col152 + "_cap"] = df[col152].clip(upper= 379400.0)

# 0.998
col39 = "MOD_ACT__pctratmttoopeaprmu6mopecrpmu12"
df[col39 + "_cap"] = df[col39].clip(upper= 2.637802)

# 0.999
col8 = "MTX_TRX_POS__pos_tkt_trx_td_prm_u6m"
df[col8 + "_cap"] = df[col8].clip(lower=0, upper= 6189.8115)

# 0.999
col26 = "MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m"
df[col26 + "_cap"] = df[col26].clip(upper=112905.56)

# 0.995
col20 = "MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_12_12_rt_u12"
df[col20 + "_cap"] = df[col20].clip(upper=3.8366303)

# 0.999
col19 = "MTX_TRX_POS__pos_tkt_trx_com_sol_prm_p6m"
df[col19 + "_cap"] = df[col19].clip(lower=0, upper=7833.3335)

# 0.998
col110 = "MTX_TRX_CANAL_PAGO_TRANSF__can_tkt_tmo_tot_pag_srv_sol_g6m"
df[col110 + "_cap"] = df[col110].clip(upper=133.00755)

# 0.99
col29 = "APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl"
df[col29 + "_cap"] = df[col29].clip(upper=46.0)

#MODEL

In [0]:
import pandas as pd
import pickle

with open("/Workspace/Users/sheryltsalazar@bcp.com.pe/02_Proyectos/06_Construccion_PD_Troncal_BHV/12_Modelamiento_pt3/indicadores_45D_noteb37//model_v2.pkl", "rb") 
as file:
    model = pickle.load(file)

In [0]:
feature = [
    'MTX_RCC_OTRA_DEUDA__rcc_mto_rdv_max_u3m_cap',
    'CLASI_EXPER_CLI__ctdempleado_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_pas_max_min_12_12_rt_u12_cap',
    'EVOL_CLI_PYM__ctdmaxdiamorau6m_cap',
    'MTX_RESUMEN_SALDO__prod_ctd_sld_act_u1m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_act_min_6_6_rt_u6m',
    'MOD_DEMO__ctdmesantiguedadempsunat_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_med_1_6_rt_u6m',
    'APP_SCORE_APROB_PYME__utl_3_rl_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_pas_min_24_24_rt_u24_cap',
    'APP_SCORE_APROB_PYME__montoade_act_max6_s_hip_rl',
    'MTX_TRX_CANAL__can_tkt_tmo_tot_sol_min_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_pct_pmtsav_pmact_24_24_rt_u24_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_tsav_med_1_6_rt_u6m_cap',
    'MOD_ACT__pctratiomtodecdeudapymertmtopasivoprmu3m_cap',
    'MTX_MOV_CARGO_PAS__isav_tkt_opec_pago_srv_prm_u3m',
    'APP_SCORE_APROB_PYME__edad_fin_cap',
    'EVOL_CLI_PYM__ctddiamoramay0u6m',
    'PASIVO_EVOL_SALD_PYM__mtoprmincrvariacionmensualprmvigsolu6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_prm_act_max_min_6_6_rt_u6m_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_12_12_rt_u12',
    'MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_min_u9m_cap',
    'MTX_RCC_PROD__rcc_tip_cond_mor_max_crnor_max_u6m_cap',
    'VIDEVAR_MTX_MORA_POND_CLI_MMGR__mtodeudaclasifriesgofactordsctosolu12_cap',
    'RNG_ACTIVIDAD_ECONOM',
    'MOD_ACT__isav_mto_opea_estvta_pym_u6m_rt_max_u12_cap',
    'MTX_RESUMEN_ACT_PAS__prod_mto_sld_fim_tsav_max_min_12_12_rt_u12_cap',
    'MTX_RESUMEN_SALDO__prod_antmin_cvi_prm_u12_cap',
    'MTX_RESUMEN_SALDO__prod_mto_sld_pai_g6m_cap',
    'MTX_MOV_ABONO_PAS__isav_tkt_opea_trnf_dol_max_u3m_cap',
    'MOD_ACT__pctratiomtoopeaprmu6mopecprmu12_cap',
    'MTX_TRX_POS__pos_tkt_trx_td_prm_u6m_cap',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_mto_tmo_tot_pag_bcp_prm_u6m_cap',
    'APP_SCORE_APROB_PYME__meses_activo_sf_bu_ma6_0_ag',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_ctd_tmo_tot_pag_bcp_frq_u6m',
    'MTX_RESUMEN_ACT_PAS__prod_ctd_mes_pas_act_max_v2_0_u12',
    'APP_SCORE_APROB_PYME__sf_num_cal_cpp24_ag',
    'MTX_MOV_ABONO_PAS__isav_flg_opea_nhab_ncts_dol_max_u12',
    'MTX_RESUMEN_ACT_PAS__prod_pct_fm_pmpas_med_12_12_rt_u12_cap',
    'MTX_TRX_POS__pos_tkt_trx_com_sol_prm_p6m_cap',
    'APP_SCORE_APROB_PYME__meses_pmsavmf_24_100_rl',
    'MTX_RCC_PROD__rcc_pct_deu_defc_max_u6m',
    'MTX_TRX_CANAL_PAGO_TRANSF__can_tkt_tmo_tot_pag_srv_sol_g6m_cap',
    'APP_SCORE_APROB_PYME__atrasomax_crnenr_12_rl_cap',
]

In [0]:
# 3. Preparar tus variables X (features)
# Asegúrate de seleccionar las mismas variables con las que entrenaste el modelo
X = df[feature]

# 4. Predecir la PD
y_pred = model.predict_proba(X)[:, 1]

# 5. Agregar la PD a tu base de datos
df['PD'] = y_pred

# 6. Verificar resultados
print(df[['PD']].head())
print(f"\nEstadísticas de PD:")
print(df['PD'].describe())

In [0]:
# df ya es un DataFrame de Pandas
result = df

# Calcular chunk_size
chunk_size = int((len(result) /1* 0.1)) ## 21 es porque son 21 meses en la base
round_size = (10**int(np.log10(chunk_size))) // 2
chunk_size = int(np.ceil(chunk_size / round_size) * round_size)

# Obtener los valores únicos de codmes (en Pandas)
result_codmes = result['codmes'].unique()

for codmes in result codmes:
    try
        output_path =f'/Workspace/Users/sherlytsalazar@bcp.com.pe/ 02_Proyectos/06_Construccion_PD_Troncal_BHV/ 15_Actualizar_PD_Hasta202601/ bd_universo_pd_bhv_troncal_202506_Valid_Impl_PDw/gen_{codmes}'
        os.makedirs(output_path, exist_ok=True)
        
        print(f"Procesando el codmes: {codmes}")

        # Filtrar por codmes (en Pandas)
        df_codmes = result[result['codmes']==codmes]

        # Ordenar el DataFrame
        pddf = df_codmes.sort_values(by=['codmes','codclaveunicocli']) I

        # Guardar en chunks usando tu función
        pd_write_in_chunks(df=pddf,
                        directory_path=output_path,
                        chunk_size=chunk_size,
                        chunk_name='data_partition',
                        file_format='parquet',
                        verbose=True,
                        index=False)

        print(f"Se ha terminado de procesar el codmes: {codmes}")
    except Exception as e: print(f"Error al procesar el codmes: {codmes} debido a {e}")

In [0]:
df_fin = pd_read_chunks(
    directory_path="/Workspace/Users/sherlytsalazar@bcp.com.pe/02_Proyectos/ 06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/bd_universo_pd_bhv_troncal_202506_Valid_Impl_PDw",file_format='parquet', verbose=True).sort_values(by=["codmes", "codclaveunicocli"], ascending=[True, True])

In [0]:
df_fin

In [0]:
from pyspark.sql.functions import expr
import math
df_fin['XB'] - np.log(df_fin['PD'] / (1 - df_fin['PD']))
df_fin['XB_CAL']- df_fin.eval(' -0.6916726371672072 + 0.8302316543993986 * XB')
df_fin['PD_CAL'] - df_fin['XB_CAL'].apply(lambda x: 1/(1+math.exp(-x)))

In [0]:
df_fin['codmes'] - df_fin['codmes'].astype(int)

In [0]:
df_fin

In [0]:
# df ya es un DataFrame de Pandas result = df fin

# Calcular chunk_size
chunk_size = int((len(result) /1*0.1))
round_size = (10**int(np.log10(chunk_size))) // 2
chunk_size = int(np.ceil(chunk_size / round_size) * round_size)

# Obtener los valores únicos de codmes (en Pandas)
result_codmes = result['codmes'].unique()

for codmes in result_codmes:
    try:
        output_path = f'/Workspace/Users/sherlytsalazar@bcp.com.pe/02_Proyectos/ 06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/ bd_universo_pd_bhv_troncal_202506_Valid_Impl_PDw_Cal/gen_{codmes}'
        os.makedirs(output_path, exist_ok=True)

        print(f"Procesando el codmes: {codmes}")

        # Filtrar por codmes (en Pandas)
        df_codmes = result[result['codmes']==codmes]

        # Ordenar el DataFrame
        pddf = df_codmes.sort_values(by-['codmes', 'codclaveunicocli'])

        # Guardar en chunks usando tu función
        pd_write_in_chunks(df=pddf,
                        directory_path=output_path,
                        chunk_size=chunk_size,
                        chunk_name='data_partition',
                        file_format='parquet',
                        verbose=True,
                        index=False)

        print(f"Se ha terminado de procesar el codmes: {codmes}")
    except Exception as e:
        print(f"Error al procesar el codmes: {codmes} debido a {e}")

In [0]:
df_ff - pd_read_chunks(
    directory_path="/Workspace/Users/sherlytsalazar@bcp.com.pe/02_Proyectos/ 06_Construccion_PD_Troncal_BHV/15_Actualizar_PD_Hasta202601/ bd_universo_pd_bhv_troncal_202506_Valid_Impl_PDw_Cal",file_format-'parquet',
    verbose-True).sort_values(by-["codmes", "codclaveunicocli"], ascending-[True, True])

In [0]:
df_ff